In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%cd /content
import shutil
from pathlib import Path
ROOT = Path("/content/Idea_agentic")
if ROOT.exists():
    shutil.rmtree(ROOT)


In [ ]:
!git clone https://github.com/vqvbku24/Idea_agentic.git /content/Idea_agentic
!git -C /content/Idea_agentic checkout main
!git -C /content/Idea_agentic status --short


In [ ]:
import os
import shutil
from pathlib import Path

ROOT = Path("/content/Idea_agentic")
DRIVE = Path("/content/drive/MyDrive/Idea_agentic")
DATA = DRIVE / "data"

# Link only large private inputs/cache. Never link generated results into the Git tree.
def link_directory(relative):
    local = ROOT / relative
    source = DRIVE / relative
    if not source.exists():
        raise FileNotFoundError(f"Missing Drive directory: {source}")
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, local, target_is_directory=True)

link_directory("data/raw")
link_directory("data/cache")
print("Linked data/raw and data/cache")


In [ ]:
import shutil
import zipfile
from pathlib import Path

OUT = ROOT / "results/real"
OUT.mkdir(parents=True, exist_ok=True)

def find_drive_file(name):
    candidates = [p for p in DRIVE.rglob(name) if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"Missing {name} under {DRIVE}")
    # Prefer the directly exported canonical path, then newest file.
    direct = DRIVE / "results/real" / name
    if direct.exists():
        return direct
    return max(candidates, key=lambda p: p.stat().st_mtime)

for name in ("data_bundle_x2.zip", "data_manifest.json", "view_manifest.json"):
    source = find_drive_file(name)
    shutil.copy2(source, OUT / name)
    print("Copied", source)

with zipfile.ZipFile(OUT / "data_bundle_x2.zip") as bundle:
    bundle.extractall(ROOT)

# Retrieval needs French distractor metadata; it is not paper text and is kept outside the bundle.
for name in ("english_distractors_metadata.jsonl", "french_distractors_metadata.jsonl"):
    source = find_drive_file(name)
    target = ROOT / "data/processed" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

# Compare against the local manifest, not against a nested results/real/real copy.
shutil.copy2(OUT / "view_manifest.json", OUT / "view_manifest.local.json")
for path in (OUT / "data_manifest.json", OUT / "view_manifest.json", OUT / "papers_x2.jsonl"):
    print("Present:", path, path.exists())


In [ ]:
%cd /content/Idea_agentic
!pip install -q sentence-transformers pymupdf scipy numpy sentencepiece


In [ ]:
import os
os.environ["HF_HOME"] = "/content/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/content/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/content/huggingface"

required = [
    ROOT / "results/real/data_manifest.json",
    ROOT / "results/real/view_manifest.json",
    ROOT / "results/real/papers_x2.jsonl",
    ROOT / "results/real/development_papers_x2.jsonl",
    ROOT / "results/real/french_distractors_native_x2.jsonl",
    ROOT / "results/real/french_distractors_native.jsonl",
    ROOT / "data/processed/english_distractors_metadata.jsonl",
    ROOT / "data/processed/french_distractors_metadata.jsonl",
    ROOT / "data/raw",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))
print("All required inputs are present")


In [ ]:
%cd /content/Idea_agentic
!python scripts/colab_run_manifest.py --ref main


In [ ]:
import subprocess
from datetime import datetime, timezone

LOG_DIR = ROOT / "results/real/colab_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / f"retrieval_v2_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.log"
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        ["python", "-u", "scripts/run_retrieval_v2.py"],
        cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    code = process.wait()
if code != 0:
    raise RuntimeError(f"Retrieval failed with exit code {code}; see {log_path}")
print("Retrieval complete:", log_path)


In [ ]:
# Colab Run 2: Full retrieval across all rankers (BM25, Dense BGE-M3, Hybrid RRF, DocTranslateBM25)
%cd /content/Idea_agentic
!python -u scripts/run_retrieval_colab_full.py


In [ ]:
%cd /content/Idea_agentic
!python -u scripts/finalize_x2_audit.py


In [ ]:
%cd /content/Idea_agentic
!python -m unittest discover -s tests -p "test*.py" -v
!python -m compileall -q src scripts


In [ ]:
import shutil
from pathlib import Path

SOURCE = ROOT / "results/real"
TARGET = DRIVE / "results/real-colab/real"
TARGET.mkdir(parents=True, exist_ok=True)

patterns = [
    "run_meta_*.json", "retrieval_metrics.csv", "retrieval_v2_diagnostics.md",
    "dense_reference_x2.csv", "translations_x2.jsonl", "blind_leak_checks_x2.csv",
    "data_manifest.json", "view_manifest.json", "run_meta_x2.json",
    "final_run.md", "leak_fail_cases_x2.md", "candidate_rule_v2_1_review.md",
    "extraction_review_x2.md", "residual_defects_x2.md", "defect_counts_x2.json",
    "retrieval_full_metrics.csv", "retrieval_full_report.md", "exploratory_report.md",
]
for pattern in patterns:
    for source in SOURCE.glob(pattern):
        shutil.copy2(source, TARGET / source.name)
for source in (SOURCE / "colab_logs").glob("*.log"):
    log_target = TARGET / "colab_logs"
    log_target.mkdir(exist_ok=True)
    shutil.copy2(source, log_target / source.name)
print("Persisted output to", TARGET)
print("Timestamped manifests:", [p.name for p in TARGET.glob("run_meta_*.json")])
